In [ ]:
import casadi.casadi as cs
import numpy as np

# System modeling
dt = 0.1 

# Define state variables
s = cs.MX.sym('s')
v_s = cs.MX.sym('v_s')
d = cs.MX.sym('d')
v_d = cs.MX.sym('v_d')
x = cs.vertcat(s, v_s, d, v_d)

# Define process noise
w_vs = cs.MX.sym('w_vs')
w_vd = cs.MX.sym('w_vd')
w = cs.vertcat(w_vs, w_vd)

# Lateral motion parameters based on Ornstein-Uhlenbeck process
k_restore = 0.05
k_damp = 0.9
k_vel = 0.1
v_target = 10.0

# Next state prediction equations
s_next = s + v_s * dt
v_s_next = v_s + k_vel * (v_target - v_s) * dt + w_vs
d_next = d + v_d * dt
v_d_next = k_damp * v_d - k_restore * d + w_vd

x_next = cs.vertcat(s_next, v_s_next, d_next, v_d_next)

# CasADi functions and Jacobians
f_dynamics = cs.Function('f_dynamics', [x, w], [x_next])
A_jac = cs.Function('A_jac', [x, w], [cs.jacobian(x_next, x)])
G_jac = cs.Function('G_jac', [x, w], [cs.jacobian(x_next, w)])

# Process noise covariance
Q = np.diag([0.1, 0.001])

